## DNS（域名系统）｜应用层 · 带过

> 2026-10-11 · 来源：学习对话

- **我的理解**：先带过 DNS / HTTP，此后不再重点投入（用户 2026-10-11 决定）。
- **机制层**：
  - 域名层次：根 → 顶级域 TLD → 权限域 → 主机，写作 `www.example.com.`（末尾的点＝根）。
  - 四类服务器：根域名服务器 / 顶级域名服务器 / 权限域名服务器 / **本地**域名服务器。
  - 查询方向：**主机 → 本地 ＝ 递归**；**本地 → 根 / 顶级 / 权限 ＝ 迭代**（本地替主机跑完全程）。
  - 高速缓存：主机与本地域名服务器都缓存近期解析结果，减少查询次数。
- **设计动机**：IP 是定长数字，难记且可能变；DNS 提供分层的、去中心化的「名字 → IP」映射。
- **项目映射**：Linux 侧 `/etc/resolv.conf` + `nsswitch.conf`，工具 `dig` / `nslookup`。**边界**：408 只考查询过程与服务器角色，不考 DNSSEC / DoH / 根区细节。
- **考试侧重点**：给一次访问，问「哪一段是递归、哪一段是迭代」「共发生几次查询」。
- **易错点**：把递归 / 迭代方向记反（**递归永远是主机↔本地那一段**）；漏掉「本地域名服务器」这个角色。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-12


## HTTP / WWW｜应用层 · 带过

> 2026-10-11 · 来源：学习对话

- **我的理解**：随 socket 一起带过，后续不再重点投入。
- **机制层**：
  - HTTP 建于 **TCP** 之上，默认端口 **80**（HTTPS 443），**无状态**。
  - 报文：请求行 / 头 / 体；响应行 / 头 / 体。
  - 方法：GET / POST / HEAD / PUT / DELETE（GET 幂等，POST 非幂等）。
  - 状态码：1xx 信息 / 2xx 成功（200）/ 3xx 重定向（301、302）/ 4xx 客户端错（404）/ 5xx 服务器错（500）。
  - 连接：非持久（HTTP/1.0，每个对象一个连接，需 2RTT）vs 持久（HTTP/1.1，keep-alive）。
  - Cookie：弥补无状态，服务器 → 客户端保存一小段状态。
- **设计动机**：无状态简化服务器；持久连接省去每请求建连开销；Cookie 把状态下放到客户端。
- **项目映射**：浏览器 DevTools / `curl` 可直接观察报文。**边界**：408 不考 HTTP/2 多路复用、TLS 握手细节。
- **考试侧重点**：状态码含义；持久 vs 非持久连接的时间 / RTT 计算（带过级，了解即可）。
- **易错点**：把 3xx 当成错误（实为重定向）；把 HTTP 说成「非可靠传输」（可靠性由下层 TCP 提供）。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-12


## DNS 递归 / 迭代 辨析（补讲）

> 2026-10-11 · 来源：学习对话（用户反馈「没看懂」）

- **我的理解**（原话）：「回答我dns的迭代和递归没看懂」。
- **判据（一句话）**：递归 = 被问者**负责到底**，只回最终答案；迭代 = 被问者**只指路**，回「下一个该问谁」。
- **方向的唯一锚点**：**主机 → 本地 ＝ 递归**（主机只发 1 次、只收 1 次）；**本地 → 根 / 顶级 / 权限 ＝ 迭代**（本地自己一问一答多轮）。
- **「本地」是枢纽**：它**收**递归请求、**发**迭代请求——同一台机器两头都干。所以别按「机器」分递归/迭代，要按「谁被问、谁负责到底」分。
- **为什么这样设计**：递归让主机端极简（只会发一次请求、等一个答案）；迭代让根服务器极轻（只指路，不替全球递归，才扛得住流量）。重活推给离用户近的本地服务器与各级缓存。
- **易错点**：把「本地 → 根」当成递归（**错，是迭代**）；以为「递归」是「服务器一级级往下替我问」——那恰恰是**迭代**在做的事。
- **复习计划**：R1 @ 2026-10-12


## HTTP 代理：请求行为什么要重写，四个头为什么必须补

> 2026-10-22 · 来源：CSAPP 项目复习（`modules/408/CSAPP.md`） §7.8（实践仓 `learning-408/csapp/proxylab/proxy.c`，目前是 14 行空壳）
客户端发给代理的是**绝对 URL**：`GET http://host[:port]/path HTTP/1.1`；代理转发给服务器时要改成**相对路径** + 补头：
`GET /path HTTP/1.0` + `Host: host` + `User-Agent: …` + `Connection: close` + `Proxy-Connection: close` + 空行

**为什么用 HTTP/1.0 + `Connection: close`**：让「连接关闭」等于「响应结束」，省掉解析 `Content-Length` / chunked 的麻烦 —— 这是本 lab 简化协议的关键设计决策。

**工程三坑**：① 不能用 `csapp.c` 里的**大写封装**（`Open_clientfd` 等出错就 `exit` 掉整个进程），代理是长跑进程，必须自己检查返回值并只关当前连接；② I/O 必须用 RIO 处理短读；③ 必须 `Signal(SIGPIPE, SIG_IGN)`，否则客户端断开后再写会被杀死。

**并发**：每连接一线程 + `pthread_detach`；`connfd` **不能传 `&connfd`**（主循环会覆盖它），要传值或每次 Malloc 一份。

**408 对应**：HTTP 报文格式、状态码、**Web 缓存与代理服务器**（真考点）；socket 编程不考。
